In [1]:
import seaborn as sns
import warnings
import matplotlib.pyplot as plt
from sympy.utilities.exceptions import ignore_warnings
import simres.operators as op


warnings.filterwarnings("ignore")

import importlib
import simres.expr
importlib.reload(simres.expr)
from simres.expr import *

import simres.operators
importlib.reload(simres.operators)
from simres.operators import *

In [2]:
# 1. 初始化执行器
alpha_pool=[]
executor = AlphaExecutor(data_dir='/Users/qiuyucheng/Documents/Alpha4/data/20251231',
                         alpha_dir='/Users/qiuyucheng/Documents/Alpha4/alphas/20251231')
# 这是加载原始数据
executor.load_all_data()
# 这是加载因子的结果
executor.load_all_simres()
# 加载alpha到alpha_matrix
executor.load_all_alphas()


--- 正在初始化数据引擎 ---
已加载字段: [high] | 形状: (1326, 2674)
已加载字段: [close] | 形状: (1326, 2674)
已加载字段: [low] | 形状: (1326, 2674)
已加载字段: [csi_500_amount] | 形状: (1326, 2674)
已加载字段: [amount] | 形状: (1326, 2674)
已加载字段: [csi_500_low] | 形状: (1326, 2674)
已加载字段: [datestr] | 形状: (2674,)
已加载字段: [csi_500_open] | 形状: (1326, 2674)
已加载字段: [stock_list] | 形状: (1326,)
已加载字段: [csi_500_weight] | 形状: (1326, 2674)
已加载字段: [csi_500_volume] | 形状: (1326, 2674)
已加载字段: [csi_500_high] | 形状: (1326, 2674)
已加载字段: [industry] | 形状: (1326, 2674)
已加载字段: [volume] | 形状: (1326, 2674)
已加载字段: [csi_500_close] | 形状: (1326, 2674)
已加载字段: [open] | 形状: (1326, 2674)
已注入自定义算子: ['at_mask', 'at_nan2zero', 'at_zero2nan', 'cs_booksize', 'cs_group_quantile', 'cs_indneut', 'cs_rank', 'cs_zscore', 'ts_correlation', 'ts_delay', 'ts_delta', 'ts_fill', 'ts_kurtosis', 'ts_max', 'ts_mean', 'ts_min', 'ts_ols', 'ts_rank', 'ts_regression', 'ts_ret', 'ts_skewness', 'ts_std', 'ts_sum', 'ts_zscore']
--- 引擎就绪 ---

--- 正在初始化数据引擎 ---
success load alpha simres5000001

# 先看看每一个alpha表现

In [ ]:
# 最简单的方式 - 按alpha_id排序
for btresult in sorted(executor.alpha_pool, key=lambda x: x['alpha_id']):
    executor.simres_cut(btresult, '20150101', '20251231', index=None)
    corr_res = executor.corr(btresult)

#  测试新的alpha的表现(sharpe/ret/tvr/dd/correlation)

In [ ]:
# 如果sr 、ret、 dd表现较好，且corr较低，考虑加入因子库
expr='ts_correlation(high/low, csi_500_volume, 5)'
alpha = executor.evaluate(f'cs_booksize(cs_rank(at_mask({expr},ts_fill(csi_500_weight)>0))-0.5)')
btresult=executor.backtest(alpha)
executor.simres_cut(btresult,'20150101','20251231')
corr_df=executor.corr(btresult)
alpha_pool.append(btresult)

# 线性因子combo

In [3]:
# 1. 定义你想要的alpha因子序号列表
alpha_list = list(range(1, 100))
# 2. 初始化存放回测结果的池子
alpha_pool_for_combo_liner = []
# 3. 使用循环读取alpha、跑回测并存入池子
for alpha_idx in alpha_list:
    # 动态从矩阵中读取对应的alpha因子
    current_alpha = executor.alpha_matrix[alpha_idx]
    # 跑回测
    bkresult = executor.backtest(current_alpha)
    # 将回测结果加入combo池
    alpha_pool_for_combo_liner.append(bkresult)

In [4]:
corr_matrix = np.corrcoef([x['net_ret'] for x in alpha_pool_for_combo_liner])
corr_df = pd.DataFrame(corr_matrix, index=alpha_list, columns=alpha_list)
corr_df.to_csv('alpha_correlation_matrix.csv')

In [ ]:
# 看看combo池中alpha的相关性;
# 2. 设置绘图风格
sns.set_theme(style="white")
# 3. 绘制热力图
plt.figure(figsize=(10, 8))
sns.heatmap(np.corrcoef([item['net_ret'] for item in alpha_pool_for_combo_liner]),
            annot=True,       # 显示具体数值
            fmt=".2f",        # 保留两位小数
            cmap='coolwarm',  # 冷暖色调，蓝色负相关，红色正相关
            center=0,         # 设置中心点为 0
            linewidths=.5)    # 格子间距

plt.title('Factor Correlation Matrix')
plt.show()

这里的combo是每一个经过截面rank、中性化等处理之后的alpha相加得到的

选取的alpha的夏普率都是低于1的，经过等权线性combo之后，效果变得出奇的好；

In [ ]:
# 线性combo
# 1. 先用列表中的第一个因子初始化 combo_alpha，并做好缺失值处理
combo_alpha = op.at_nan2zero(executor.alpha_matrix[alpha_list[0]])
for alpha_idx in alpha_list[1:]:
    alpha = executor.alpha_matrix[alpha_idx]
    combo_alpha = combo_alpha + op.at_nan2zero(alpha)
# 如果你要对combo因子做cs_rank等等清理：
# executor.context['linear_raw_combo'] = combo_alpha
# linear_combo_cleaned = executor.evaluate(
#     'at_nan2zero(cs_booksize(cs_rank(at_mask(linear_raw_combo, ts_fill(csi_500_weight)>0))-0.5))'
# )
# linear_combo_bkresult = executor.backtest(linear_combo_cleaned)
# combo_corr = executor.corr(linear_combo_bkresult)
# executor.simres_cut(linear_combo_bkresult,'20190101','20251231',index=None)

bkresult = executor.backtest(combo_alpha)
executor.simres_cut(bkresult, '20150101', '20251231', index=None)
corr=executor.corr(bkresult)

### 线性combo因子存放

In [ ]:
enddate, alpha_id = '20251231', "5000036"

# 尝试 1：最常见的情况，框架在 context 里有标准对齐的列表
true_index = executor.context['stock_list']
true_columns = executor.context['datestr']

# 2. 将 combo_alpha 直接作为 numpy 传入（确保回测逻辑和之前完全一致）
# 我们直接把之前跑出 2.0 的真实回测结果（bkresult）拿过来切片、保存！
# 这样就不用担心标签对不对了，因为 btresult 本身就是对的！
btresult = executor.backtest(combo_alpha)
executor.simres_cut(btresult, '20150101', '20251231', index=None)
btresult['alpha_id'] = str(alpha_id)

# 3. 封装 DataFrame 用于保存
combo_alpha_df = pd.DataFrame(
    combo_alpha,
    index=true_index,
    columns=true_columns
)

# 4. 保存矩阵和回测结果
matrix_path = f"../alphas/{enddate}/matrix/{alpha_id}"
os.makedirs(os.path.dirname(matrix_path), exist_ok=True)
combo_alpha_df.to_parquet(matrix_path)

simres_path = f"../alphas/{enddate}/simres/{alpha_id}.pkl"
os.makedirs(os.path.dirname(simres_path), exist_ok=True)
with open(simres_path, "wb") as f:
    pickle.dump(btresult, f)

print(f"因子 {alpha_id} 及其 2.0 净值曲线已强制同步保存！")

### ML进行非线性因子combo

In [ ]:
# alpha_ml算出来的是numpy矩阵的因子值；
# alpha_pool_for_combo_ml中全都是这些numpy矩阵
# 在alpha_ml这些矩阵中，只有中证500的股票是有值的！！！这个需要在后面筛选的时候用到
with open ('csi500.txt', 'r') as f:
    alpha_list=f.read().split('\n')
# 非线性因子combo池
alpha_pool_for_combo_ml = []
# 根据第一步alpha的回测,挑选alpha进行combo
selected_alpha_indices = [0,6,18,19,24,48,66]
for idx in selected_alpha_indices:
    expr = alpha_list[idx]
    print(f"正在解析因子 [{idx}]: {expr}")
    # 核心修改：去掉外层的 at_nan2zero 【如果加上这个,因为缺失值去填充0会对后面的机器学习产生误导！】
    alpha_ml = executor.evaluate(f'cs_booksize(cs_rank(at_mask({expr},ts_fill(csi_500_weight)>0))-0.5)')
    # 加入到机器学习专用的池子中
    alpha_pool_for_combo_ml.append(alpha_ml)

### 构建label

In [ ]:
# 提取原始收益率 NumPy 数组 (结构: 1326只股票 x 2674个交易日)
# raw_ret也都是numpy矩阵,计算的是(VWAPT+1  /  VWAPT）- 1
raw_ret = executor.context['ret1']

# 1. 创建一个全 NaN 的空矩阵，形状与原收益率矩阵完全一致
forward_ret = np.full_like(raw_ret, np.nan, dtype=np.float64)

# 2. 核心逻辑：因为列是时间(axis=1)，我们要将 T+2 日的数据向前移给 T 日
# 也就是把第 2 列开始到最后一列的数据，赋值给从第 0 列开始的位置
forward_ret[:, :-2] = raw_ret[:, 2:]
# 3 可选：将label绝对收益率转为截面相对收益率
forward_rank = op.cs_rank(forward_ret)

print("--- [Cell 1] Label 矩阵构建完成 ---")
print(f"原始收益率矩阵形状 (raw_ret): {raw_ret.shape} -> (股票数, 时间轴)")
print(f"平移后 Label 矩阵形状 (forward_ret): {forward_ret.shape}")
print("注意：每一行的最后 2 列已自动留空(NaN)，确保无未来数据泄露。")

### 将alpha与label矩阵展平对齐,我们在这里要对ret1中进行中证500筛选;

In [ ]:
# 在构建 ml_panel_data 时，立即记录 day_idx
num_stocks = 1326
total_days = 2674

# 构建DataFrame时就带上 day_idx
ml_panel_data = pd.DataFrame({
    'factor_01': alpha_pool_for_combo_ml[0].flatten(),
    'factor_02': alpha_pool_for_combo_ml[1].flatten(),
    'factor_03': alpha_pool_for_combo_ml[2].flatten(),
    'factor_04': alpha_pool_for_combo_ml[3].flatten(),
    'factor_05': alpha_pool_for_combo_ml[4].flatten(),
    'factor_06': alpha_pool_for_combo_ml[5].flatten(),
    'factor_07': alpha_pool_for_combo_ml[6].flatten(),
    'target_label': forward_ret.flatten(),
    'day_idx': np.repeat(np.arange(total_days), num_stocks)  # ✅ 改用 repeat
})

# 然后再做 dropna 和过滤
ml_panel_data = ml_panel_data.dropna(subset=['target_label'])
ml_panel_data = ml_panel_data[~(
    (ml_panel_data['factor_01'] == 0) &
    (ml_panel_data['factor_02'] == 0) &
    (ml_panel_data['factor_03'] == 0) &
    (ml_panel_data['factor_04'] == 0) &
    (ml_panel_data['factor_05'] == 0) &
    (ml_panel_data['factor_06'] == 0) &
    (ml_panel_data['factor_07'] == 0)
)]

print("--- [成功] 中证500专属机器学习面板数据构建完成 ---")
print(f"过滤非成分股后，剩余纯净中证500样本总行数: {len(ml_panel_data)}")
print("\n数据前 5 行预览：")
display(ml_panel_data.head())

# 快速验证：检查前几天的 day_idx 是否正确
print(ml_panel_data[['day_idx']].tail(20))
# 应该看到：0,0,0,... (第一个股票的所有天数)
# 或者根据你的原始数据结构，可能是 0,1,2,...（取决于展平顺序）

### 定义机器学习模型

In [ ]:
import lightgbm as lgb
import xgboost as xgb

# 1. 模型切换开关: 可选 'lightgbm' 或 'xgboost'
MODEL_TYPE = 'lightgbm'

# 2. 核心超参数（没有验证集，直接固定迭代轮数）
NUM_ROUNDS = 100  # 树的数量（通常 50-150 即可，过多易过拟合）

LGB_PARAMS = {
    'objective': 'regression',
    'learning_rate': 0.05,
    'max_depth': 4,
    'num_leaves': 7,
    'min_data_in_leaf': 1000,
    'verbosity': -1,
    'random_state': 42
}

XGB_PARAMS = {
    'objective': 'reg:squarederror',
    'learning_rate': 0.05,
    'max_depth': 4,
    'min_child_weight': 1000,
    'tree_method': 'hist',
    'random_state': 42
}

def train_and_predict(X_train, y_train, X_pred):
    """ 统一的训练和纯样本外预测接口 """
    if MODEL_TYPE == 'lightgbm':
        train_data = lgb.Dataset(X_train, label=y_train)
        model = lgb.train(LGB_PARAMS, train_data, num_boost_round=NUM_ROUNDS)
        return model.predict(X_pred)

    elif MODEL_TYPE == 'xgboost':
        model = xgb.XGBRegressor(**XGB_PARAMS, n_estimators=NUM_ROUNDS)
        model.fit(X_train, y_train, verbose=False)
        return model.predict(X_pred)

### 滚动窗口训练（1000天训练，200天测试）

In [ ]:
import numpy as np
import pickle  # 添加导入

# 1. 基础维度定义
total_days = 2674
num_stocks = 1326


# 2. 严格遵循要求的窗口参数
TRAIN_WINDOW = 1500  # 训练
PRED_WINDOW = 100   # 预测

start_pred_day = TRAIN_WINDOW
all_preds_flattened = np.full(num_stocks * total_days, np.nan, dtype=np.float64)

features = ['factor_01', 'factor_02', 'factor_03', 'factor_04', 'factor_05', 'factor_06', 'factor_07']
target = 'target_label'

# ✅ 新增：用于保存所有模型
models_list = []

# 3. 极简滚动窗口循环
current_start = start_pred_day

#         将绝对收益率转为——截面百分比排名或者截面Z-score排名
#
#  将绝对收益率转为截面百分比排名 (Rank Percentile)
# ml_panel_data['target_label'] = ml_panel_data.groupby('day_idx')['target_label'].rank(pct=True)
# ml_panel_data['target_label'] = ml_panel_data['target_label'].fillna(0.5)


# 1. 计算每个交易日内部的均值和标准差
mean_target = ml_panel_data.groupby('day_idx')['target_label'].transform('mean')
std_target = ml_panel_data.groupby('day_idx')['target_label'].transform('std')
# 2. 转换绝对收益率为截面 Z-Score
ml_panel_data['target_label'] = (ml_panel_data['target_label'] - mean_target) / std_target
# 去极值
ml_panel_data['target_label'] = ml_panel_data['target_label'].clip(-3, 3)
# 填充0
ml_panel_data['target_label'] = ml_panel_data['target_label'].fillna(0.0)


while current_start < total_days:
    current_end = min(current_start + PRED_WINDOW, total_days)

    # 划分当前批次的时序区间
    train_part = ml_panel_data[(ml_panel_data['day_idx'] >= current_start - TRAIN_WINDOW) & (ml_panel_data['day_idx'] < current_start)]
    pred_part = ml_panel_data[(ml_panel_data['day_idx'] >= current_start) & (ml_panel_data['day_idx'] < current_end)]

    if len(pred_part) == 0:
        current_start += PRED_WINDOW
        continue

    print(f"滚动训练中: 预测第 {current_start} ~ {current_end-1} 天... ", end="")

    # ✅ 修改：显式训练模型并保存
    if MODEL_TYPE == 'lightgbm':
        import lightgbm as lgb
        train_data = lgb.Dataset(train_part[features], label=train_part[target])
        model = lgb.train(LGB_PARAMS, train_data, num_boost_round=NUM_ROUNDS)
        y_pred_chunk = model.predict(pred_part[features])

        # 保存模型和对应的日期范围
        models_list.append({
            'model': model,
            'train_start_day': current_start - TRAIN_WINDOW,
            'train_end_day': current_start - 1,
            'predict_start_day': current_start,
            'predict_end_day': current_end - 1
        })

    elif MODEL_TYPE == 'xgboost':
        import xgboost as xgb
        model = xgb.XGBRegressor(**XGB_PARAMS, n_estimators=NUM_ROUNDS)
        model.fit(train_part[features], train_part[target], verbose=False)
        y_pred_chunk = model.predict(pred_part[features])

        # 保存模型和对应的日期范围
        models_list.append({
            'model': model,
            'train_start_day': current_start - TRAIN_WINDOW,
            'train_end_day': current_start - 1,
            'predict_start_day': current_start,
            'predict_end_day': current_end - 1
        })

    # 还原放回一维大槽位
    all_preds_flattened[pred_part.index] = y_pred_chunk
    print("[OK]")

    current_start += PRED_WINDOW

# 4. 瞬间 Reshape 还原回原始的二维 Alpha 因子矩阵结构 (股票数 x 时间轴)
ml_combo_alpha_matrix = all_preds_flattened.reshape(num_stocks, total_days)

# ✅ 5. 保存所有模型权重
with open('ml_combo/lightgbm_models_weights.pkl', 'wb') as f:
    pickle.dump(models_list, f)

print(f"\n--- [成功] 已保存 {len(models_list)} 个 LightGBM 模型权重 ---")
print(f"文件保存为: lightgbm_models_weights.pkl")
print(f"矩阵最终形状: {ml_combo_alpha_matrix.shape}")
print("\n--- [成功] 滚动非线性复合 Alpha 矩阵构建完成！ ---")

 ### 对ml_combo_alpha_matrix进行回测

In [ ]:
# 注入context,才能用evaluate对其进行解析
executor.context['ml_raw_combo'] = ml_combo_alpha_matrix
# 对原生的ml_combo_alpha_matrix进行中证500筛选,截面rank,中性化，多空组合，缺失值填充操作；
ml_combo_alpha_cleaned = executor.evaluate(
    'at_nan2zero(cs_booksize(cs_rank(at_mask(ml_raw_combo, ts_fill(csi_500_weight)>0))-0.5))'
)
ml_combo_alpha_bkresult = executor.backtest(ml_combo_alpha_cleaned)
executor.simres_cut(ml_combo_alpha_bkresult,'20150101','20251231')